# Module 5 – Embeddings, Indexing & Retrieval

## 📍 Where We Are in the Pipeline

```
Document → Extract → Chunk → EMBED → INDEX → RETRIEVE → Generate
                     ✅ M4    🔵 NOW  🔵 NOW  🔵 NOW
```

**This module covers THREE critical pipeline stages:**
1. **🧮 EMBED** – Convert text chunks to 3072-dimensional vectors
2. **📦 INDEX** – Store vectors in Azure AI Search
3. **🔎 RETRIEVE** – Find relevant chunks for user queries

---

## Learning Outcomes

By the end of this module, you will be able to:
- Generate embeddings using `text-embedding-3-large`
- Design index schemas for RAG workloads with vector fields
- Create and populate an Azure AI Search index (Push model)
- Implement text, vector, and hybrid search
- Configure semantic ranking for improved relevance
- Select the right retrieval pattern for different use cases
- **Use Agentic Retrieval for complex multi-part questions (Preview)**

---

## 📋 Dataset Reminder

We're working with **Israel M1 Metro Line** station documents:
- **metro-s36.pdf**: Station 36 (שדרות הציונות) specification
- Contains: Station specs, passenger forecasts, land use tables, maps, figures
- Languages: Hebrew + English

Our queries will focus on Metro station information!

---

## ⏱️ Estimated Time: ~2.5 hours

| Section | Time |
|---------|------|
| Part 0: Setup & Load Chunks | 10 min |
| Part 1: Embeddings | 25 min |
| Part 2: Index Creation | 25 min |
| Part 3: Search Modes | 35 min |
| Part 4: Retrieval Patterns | 35 min |
| Part 5: Agentic Retrieval (Preview) | 30 min |

---

# Part 0: Setup & Load Chunks from Module 4

First, let's load our environment and the chunks we created in Module 4.

## 📷 Our Source Document: Metro Station 36

This is the PDF we're working with - a technical specification for Metro Station 36 (שדרות הציונות):

![Metro Station 36 Document](../../data/module1-images/metro_s36_page1.png)

**Key content in this document:**
- Station location and surroundings
- Passenger capacity forecasts  
- Land use tables (ייעודי קרקע)
- Station entrance maps and figures
- Technical specifications

The challenge: How do we enable a chatbot to **accurately answer questions** about this complex document?

In [ ]:
# Cell 0.1: Install/verify dependencies
import sys
!{sys.executable} -m pip install -q azure-search-documents==11.6.0 openai python-dotenv tqdm azure-identity

In [ ]:
# Cell 0.2: Setup and load environment
import os
import sys
import json
from pathlib import Path

# Add src to path for utilities
sys.path.append(str(Path("../../src").resolve()))
from utils import load_env

# Load environment variables
env = load_env()
print("✅ Environment loaded.")

# Set project root for finding files
PROJECT_ROOT = Path("../../").resolve()

# Verify required environment variables
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_SEARCH_ENDPOINT",
]

missing = [v for v in required_vars if not env.get(v)]
if missing:
    raise ValueError(f"❌ Missing environment variables: {missing}")

print(f"   - OpenAI Endpoint: {env.get('AZURE_OPENAI_ENDPOINT', '')[:50]}...")
print(f"   - Search Endpoint: {env.get('AZURE_SEARCH_ENDPOINT', '')[:50]}...")

In [ ]:
# Cell 0.3: Load chunks from Module 4
chunks_path = PROJECT_ROOT / "modules" / "module-4-chunking" / "output" / "hybrid_chunks.json"

if not chunks_path.exists():
    raise FileNotFoundError(f"❌ Chunks file not found: {chunks_path}\n   Please complete Module 4 first.")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks from Module 4")
print(f"   Source: metro-s36.pdf (Station 36 - שדרות הציונות)")

# Analyze chunk distribution
content_types = {}
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    content_types[ct] = content_types.get(ct, 0) + 1

print(f"\n📊 Chunk Distribution:")
for ct, count in sorted(content_types.items(), key=lambda x: -x[1]):
    print(f"   {ct}: {count}")

In [ ]:
# Cell 0.4: Inspect sample chunks
print("📄 Sample Chunks from Metro Station 36:\n")

# Show one of each type
shown_types = set()
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    if ct not in shown_types:
        shown_types.add(ct)
        print(f"--- {ct.upper()} (id: {chunk['id']}) ---")
        content = chunk['content'][:300] + "..." if len(chunk['content']) > 300 else chunk['content']
        print(content)
        print(f"\nMetadata: {chunk.get('metadata', {})}")
        print("\n")
    if len(shown_types) >= 3:
        break

---

# Part 1: Embeddings

## 🎯 Learning Goal
Understand how text is converted to vectors that capture semantic meaning, enabling "search by meaning" rather than just keyword matching.

## What are Embeddings?

Embeddings are **dense vector representations** of text that capture semantic meaning:
- Similar concepts have vectors that are close together in high-dimensional space
- `text-embedding-3-large` produces **3072-dimensional** vectors
- Enable **semantic search** beyond keyword matching

### Why Embeddings Matter for Metro Documents

```
"תחנה 36"           →  [0.023, -0.156, 0.089, ..., 0.042]  (3072 floats)
"Station 36"        →  [0.021, -0.152, 0.091, ..., 0.039]  (similar!)
"שדרות הציונות"     →  [0.019, -0.148, 0.092, ..., 0.038]  (also similar!)
"pizza recipe"      →  [-0.234, 0.078, -0.156, ..., -0.089]  (very different)
```

The embedding model understands that "תחנה 36", "Station 36", and "שדרות הציונות" are all related to the same metro station - even across languages!

### How Similar is "Similar"?

| Cosine Similarity | Interpretation |
|-------------------|----------------|
| 0.8 - 1.0 | Nearly identical meaning |
| 0.5 - 0.8 | Clearly related concepts |
| 0.3 - 0.5 | Weakly related |
| < 0.3 | Unrelated |

> 💡 **Real-world insight**: Cross-lingual similarity (Hebrew↔English) typically scores 0.5-0.7, not 0.9+. This is still highly useful for retrieval!

## Lab 1.1: Initialize OpenAI Client

We use Azure OpenAI's `text-embedding-3-large` model to generate embeddings. This model:
- Handles **multilingual text** (Hebrew + English)
- Produces **3072-dimensional** vectors
- Has a context window of **8191 tokens** (~32,000 characters)

### 📷 Azure OpenAI Deployments

Make sure you have the `text-embedding-3-large` model deployed in your Azure OpenAI resource:

![OpenAI Deployments](images/openai-deployments.png)

> 📸 **TODO**: Add screenshot of Azure Portal → Azure OpenAI → Model deployments showing text-embedding-3-large

In [ ]:
# Cell 1.1: Initialize Azure OpenAI client for embeddings
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Use Entra ID authentication (keys are disabled on this resource)
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

openai_client = AzureOpenAI(
    azure_endpoint=env["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version=env.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
)

EMBEDDING_MODEL = env.get("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")
EMBEDDING_DIMENSIONS = 3072

print(f"✅ OpenAI client initialized (using Entra ID)")
print(f"   - Embedding model: {EMBEDDING_MODEL}")
print(f"   - Dimensions: {EMBEDDING_DIMENSIONS}")

## Lab 1.2: Generate a Single Embedding

Let's see what an embedding looks like. We'll embed a Hebrew sentence about Metro Station 36 and inspect the resulting vector.

In [ ]:
# Cell 1.2: Generate embedding for a single text
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    """
    Generate embedding for a single text.
    
    Args:
        text: Input text (max ~8191 tokens)
        model: Embedding model deployment name
        
    Returns:
        List of floats (3072 dimensions)
    """
    # Clean and truncate text if needed (rough estimate: 4 chars per token)
    max_chars = 8000 * 4  # ~32000 chars
    if len(text) > max_chars:
        text = text[:max_chars]
    
    response = openai_client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# Test with Metro-related text
test_text = "תחנה מספר 36 שדרות הציונות - קיבולת נוסעים צפויה 2,400 נוסעים בשעת שיא"
test_embedding = get_embedding(test_text)

print(f"✅ Generated embedding for Metro Station 36 text")
print(f"   - Input: {test_text}")
print(f"   - Input length: {len(test_text)} characters")
print(f"   - Output dimensions: {len(test_embedding)}")
print(f"   - First 5 values: {test_embedding[:5]}")

## Lab 1.3: Semantic Similarity Demo

The power of embeddings is that they capture **meaning**, not just words. Let's prove this by comparing:
- The same question in Hebrew and English
- A question and its answer
- A completely unrelated sentence

We measure similarity using **cosine similarity** (1.0 = identical, 0 = orthogonal, -1 = opposite).

In [ ]:
# Cell 1.3: Demonstrate semantic similarity with Metro content
import numpy as np

def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    a = np.array(v1)
    b = np.array(v2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test sentences - Metro Station 36 related
sentences = [
    "כמה נוסעים צפויים בתחנה 36?",                          # S1: Hebrew question about passengers
    "How many passengers are expected at Station 36?",       # S2: Same question in English
    "קיבולת נוסעים צפויה 2,400 נוסעים בשעת שיא",            # S3: Answer about passenger capacity
    "I love eating pizza on Friday nights."                  # S4: Completely unrelated
]

print("📊 Semantic Similarity Matrix (Metro Station 36 queries)\n")
print(f"{'':>5}", end="")
for i in range(len(sentences)):
    print(f"  S{i+1}  ", end="")
print("\n")

embeddings_demo = [get_embedding(s) for s in sentences]

for i, emb_i in enumerate(embeddings_demo):
    print(f"S{i+1}  ", end="")
    for j, emb_j in enumerate(embeddings_demo):
        sim = cosine_similarity(emb_i, emb_j)
        print(f" {sim:.3f} ", end="")
    print()

print("\n📝 Sentences:")
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s[:60]}..." if len(s) > 60 else f"S{i+1}: {s}")

print("\n💡 What to Look For:")
print("   - S1↔S2: Cross-lingual similarity (Hebrew↔English same question)")
print("     Typical range: 0.5-0.7 (not perfect, but clearly related)")
print("   - S1/S2↔S3: Question-Answer relationship (~0.4-0.5)")
print("   - S4 (pizza): Should be near 0 or negative (unrelated)")
print("\n🎓 Key Insight: Embeddings capture MEANING, not exact words!")
print("   Even 0.5+ similarity is useful for retrieval - we retrieve top-K,")
print("   not just perfect matches. The unrelated S4 is clearly separated.")

## Lab 1.4: Batch Embedding Generation

Embedding one text at a time is slow. In production, we process **batches of 16-20 texts** per API call for efficiency. This function handles:
- Batching for throughput
- Rate limiting to avoid API throttling
- Error handling for failed batches

In [ ]:
# Cell 1.4: Batch embedding function
from tqdm import tqdm
import time

def get_embeddings_batch(
    texts: list[str], 
    model: str = EMBEDDING_MODEL, 
    batch_size: int = 16,
    show_progress: bool = True
) -> list[list[float]]:
    """
    Generate embeddings for multiple texts in batches.
    
    Args:
        texts: List of input texts
        model: Embedding model deployment name
        batch_size: Number of texts per API call (max ~16 recommended)
        show_progress: Show progress bar
        
    Returns:
        List of embedding vectors
    """
    all_embeddings = []
    max_chars = 8000 * 4  # Token limit safety
    
    # Process in batches
    batches = [texts[i:i+batch_size] for i in range(0, len(texts), batch_size)]
    
    iterator = tqdm(batches, desc="Generating embeddings") if show_progress else batches
    
    for batch in iterator:
        # Truncate long texts
        batch_cleaned = [t[:max_chars] if len(t) > max_chars else t for t in batch]
        
        try:
            response = openai_client.embeddings.create(
                input=batch_cleaned,
                model=model
            )
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"❌ Error in batch: {e}")
            # Add empty embeddings for failed batch (handle gracefully)
            all_embeddings.extend([[0.0] * EMBEDDING_DIMENSIONS] * len(batch))
        
        # Rate limiting - be nice to the API
        time.sleep(0.1)
    
    return all_embeddings

print("✅ Batch embedding function defined")

## Lab 1.5: Generate Embeddings for All Chunks

Now let's embed all our Metro Station 36 chunks from Module 4. 

**What's happening:**
1. Each chunk's text content is sent to Azure OpenAI
2. The model returns a 3072-dimensional vector
3. We store these vectors alongside the chunk data

This is the **EMBED** stage of our pipeline!

In [ ]:
# Cell 1.5: Generate embeddings for all chunks

# Extract text content from chunks
chunk_texts = [chunk["content"] for chunk in chunks]

print(f"📊 Embedding {len(chunk_texts)} Metro Station 36 chunks...")
print(f"   - Estimated time: ~{len(chunk_texts) // 16 * 2 + 5} seconds\n")

start_time = time.time()
embeddings = get_embeddings_batch(chunk_texts, batch_size=16)
elapsed = time.time() - start_time

print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.1f}s")
print(f"   - Rate: {len(embeddings)/elapsed:.1f} embeddings/sec")
print(f"   - Each embedding: {len(embeddings[0])} dimensions")

In [ ]:
# Cell 1.6: Attach embeddings to chunks

# Create enriched chunks with embeddings
enriched_chunks = []
for i, chunk in enumerate(chunks):
    enriched_chunk = chunk.copy()
    enriched_chunk["embedding"] = embeddings[i]
    enriched_chunks.append(enriched_chunk)

print(f"✅ Created {len(enriched_chunks)} enriched chunks with embeddings")

# Verify
sample = enriched_chunks[0]
print(f"\n📄 Sample enriched chunk:")
print(f"   - id: {sample['id']}")
print(f"   - content_type: {sample.get('content_type', 'unknown')}")
print(f"   - content length: {len(sample['content'])} chars")
print(f"   - embedding dimensions: {len(sample['embedding'])}")

### ✅ Part 1 Complete!

We've successfully:
1. Generated 3072-dimensional embeddings for each chunk
2. Attached embeddings to our chunk data
3. Verified cross-lingual semantic similarity works

**What's stored in each enriched chunk:**
- `id`: Unique identifier
- `content`: The text content
- `content_type`: text, table, or figure
- `embedding`: 3072 floats representing semantic meaning
- `metadata`: Additional info (page number, etc.)

**Next**: Store these in Azure AI Search!

---

# Part 2: Azure AI Search Index

## 🎯 Learning Goal
Learn how to store embeddings in Azure AI Search and design an index schema optimized for RAG workloads.

## Azure AI Search Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Azure AI Search                          │
├─────────────────────────────────────────────────────────────┤
│  ┌──────────────────────────────────────────────────────┐  │
│  │              INDEX: module5-metro-index               │  │
│  │  ┌────────────────────────────────────────────────┐  │  │
│  │  │  DOCUMENTS (Metro Station 36 chunks)           │  │  │
│  │  │  ┌────┐ ┌────┐ ┌────┐ ┌────┐                   │  │  │
│  │  │  │text│ │text│ │table│ │fig │ ...              │  │  │
│  │  │  └────┘ └────┘ └────┘ └────┘                   │  │  │
│  │  └────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

**Key Concepts:**
- **Index**: Schema definition (like a database table) - defines what fields each document has
- **Document**: Individual item in the index (like a row) - each chunk becomes a document
- **Field**: Attribute of a document (like a column) - text, vectors, metadata
- **Vector Field**: Special field type that enables semantic (kNN) search

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Overview page

### 📷 Azure AI Search in the Portal

![Azure AI Search Overview](images/search-service-overview.png)

This is what Azure AI Search looks like in the Azure Portal. Notice the **Indexes** section where our index will be created:

## Lab 2.1: Initialize Search Client

We need two clients:
- **SearchIndexClient**: For creating/managing index schemas
- **SearchClient**: For uploading/querying documents

Both support API Key or Entra ID authentication.

In [ ]:
# Cell 2.1: Initialize Azure AI Search clients
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchableField,
    SimpleField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)

# Configuration
SEARCH_ENDPOINT = env["AZURE_SEARCH_ENDPOINT"]
SEARCH_API_KEY = env.get("AZURE_SEARCH_API_KEY")  # May be None if using Entra ID

# Use a DEDICATED index for Module 5 (separate from Module 7's pipeline index)
INDEX_NAME = "module5-metro-index"

# Create clients - try API key first, fall back to Entra ID
if SEARCH_API_KEY:
    search_credential = AzureKeyCredential(SEARCH_API_KEY)
    auth_method = "API Key"
else:
    search_credential = credential  # Use Entra ID credential from earlier
    auth_method = "Entra ID"

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential)

print(f"✅ Search clients initialized")
print(f"   - Endpoint: {SEARCH_ENDPOINT}")
print(f"   - Index name: {INDEX_NAME} (dedicated for Module 5)")
print(f"   - Auth: {auth_method}")

## Lab 2.2: Design the Index Schema

A well-designed schema is **critical** for RAG performance. Each field serves a specific purpose:

| Field | Type | Purpose | Why It Matters |
|-------|------|---------|----------------|
| `id` | string | Unique identifier (key) | Required by Azure Search |
| `content` | string | Searchable text content | BM25 keyword search |
| `content_type` | string | Type (text, table, figure) | **Filter by content type!** |
| `embedding` | vector(3072) | Semantic search vector | kNN similarity search |
| `section_header` | string | Section title | Context for LLM |
| `strategy` | string | Chunking strategy used | Debugging/analysis |
| `metadata` | string | JSON metadata | Flexible additional data |

> 🎓 **Key Design Decision**: Including `content_type` lets us filter searches to only tables, only figures, etc. This is crucial for questions like "show me the land use table".

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Indexes → module5-metro-index → Fields

### 📷 Index Schema in Portal

![Index Schema](images/search-index-schema.png)

After creating the index, you can view its schema in the Azure Portal:

In [ ]:
# Cell 2.2: Define the index schema

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters={
                "m": 4,          # Number of bi-directional links (default: 4)
                "efConstruction": 400,  # Size of dynamic list during indexing
                "efSearch": 500,        # Size of dynamic list during search
                "metric": "cosine"      # Distance metric
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-config"
        )
    ]
)

# Semantic search configuration (for L2 reranking)
semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# Define fields
fields = [
    # Key field (required)
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True
    ),
    # Content field (searchable) - supports Hebrew and English
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="standard.lucene"  # Works with both Hebrew and English
    ),
    # Content type (for filtering by chunk type)
    SimpleField(
        name="content_type",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Section header (for context)
    SearchableField(
        name="section_header",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True
    ),
    # Strategy field
    SimpleField(
        name="strategy",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Vector embedding field
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="vector-profile"
    ),
    # Metadata as JSON string
    SimpleField(
        name="metadata",
        type=SearchFieldDataType.String,
        filterable=False
    ),
]

print("✅ Index schema defined")
print(f"\n📋 Fields:")
for f in fields:
    print(f"   - {f.name}: {f.type}")

## Lab 2.3: Create the Index

This cell creates the index in Azure AI Search. The code handles three scenarios:
1. **Index doesn't exist** → Create it
2. **Index exists with correct schema** → Use it as-is
3. **Index exists with wrong schema** → Delete and recreate

> ⚠️ **Note**: If you re-run the workshop, the existing index will be reused (or recreated if schema changed).

In [ ]:
# Cell 2.3: Create or update the index

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

# Required fields in our schema
REQUIRED_FIELDS = {"id", "content", "content_type", "section_header", "strategy", "embedding", "metadata"}

# Check if index exists and has correct schema
existing_indexes = [idx.name for idx in index_client.list_indexes()]

if INDEX_NAME in existing_indexes:
    existing_index = index_client.get_index(INDEX_NAME)
    existing_field_names = {f.name for f in existing_index.fields}
    
    missing_fields = REQUIRED_FIELDS - existing_field_names
    
    if missing_fields:
        print(f"⚠️  Index '{INDEX_NAME}' exists but is missing fields: {missing_fields}")
        print("   Deleting and recreating with correct schema...")
        index_client.delete_index(INDEX_NAME)
        result = index_client.create_index(index)
        print(f"✅ Index '{result.name}' recreated with all required fields")
    else:
        print(f"✅ Using existing index '{INDEX_NAME}' (schema matches)")
        result = existing_index
else:
    # Create new index
    result = index_client.create_index(index)
    print(f"✅ Index '{result.name}' created successfully")

print(f"   - Vector search: HNSW (cosine similarity)")
print(f"   - Semantic search: Enabled (for L2 reranking)")
print(f"   - Fields: {[f.name for f in result.fields]}")

## Lab 2.4: Upload Documents (Push Model)

Azure AI Search supports two ingestion patterns:
- **Push Model**: Application uploads documents directly via SDK ← We use this
- **Pull Model**: Indexer pulls from data source (Blob, SQL, etc.)

For RAG with pre-computed embeddings, **Push** is typically better.

In [ ]:
# Cell 2.4: Prepare documents for upload

def prepare_document(chunk: dict) -> dict:
    """
    Convert a chunk to a search document.
    
    Args:
        chunk: Enriched chunk with embedding
        
    Returns:
        Document dict ready for indexing
    """
    return {
        "id": chunk["id"],
        "content": chunk["content"],
        "content_type": chunk.get("content_type", "text"),
        "section_header": chunk.get("section_header", ""),
        "strategy": chunk.get("strategy", "unknown"),
        "embedding": chunk["embedding"],
        "metadata": json.dumps(chunk.get("metadata", {}))
    }

# Prepare all documents
documents = [prepare_document(chunk) for chunk in enriched_chunks]

print(f"✅ Prepared {len(documents)} documents for upload")
print(f"\n📄 Sample document:")
sample_doc = documents[0].copy()
sample_doc["embedding"] = f"[{len(sample_doc['embedding'])} floats]"  # Truncate for display
sample_doc["content"] = sample_doc["content"][:100] + "..."
for k, v in sample_doc.items():
    print(f"   {k}: {v}")

In [ ]:
# Cell 2.5: Upload documents in batches

# Create search client for document operations
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=search_credential
)

# Upload in batches of 100 (recommended max)
batch_size = 100
total_uploaded = 0
total_failed = 0

print(f"📤 Uploading {len(documents)} Metro Station 36 chunks in batches of {batch_size}...\n")

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    try:
        result = search_client.upload_documents(documents=batch)
        succeeded = sum(1 for r in result if r.succeeded)
        failed = len(batch) - succeeded
        total_uploaded += succeeded
        total_failed += failed
        print(f"   Batch {i//batch_size + 1}: {succeeded} succeeded, {failed} failed")
    except Exception as e:
        print(f"   Batch {i//batch_size + 1}: ❌ Error - {e}")
        total_failed += len(batch)

print(f"\n✅ Upload complete: {total_uploaded} succeeded, {total_failed} failed")

In [ ]:
# Cell 2.6: Verify index population
import time

# Wait a moment for index to update
time.sleep(2)

# Get document count
results = search_client.search(search_text="*", include_total_count=True)
total_count = results.get_count()

print(f"✅ Index '{INDEX_NAME}' now contains {total_count} documents")

# Get facets by content_type
facet_results = search_client.search(
    search_text="*",
    facets=["content_type"],
    top=0
)
facets = facet_results.get_facets()

if facets and "content_type" in facets:
    print(f"\n📊 Content type distribution:")
    for facet in facets["content_type"]:
        print(f"   - {facet['value']}: {facet['count']}")

### ✅ Part 2 Complete!

We've successfully:
1. Created an Azure AI Search index with vector fields
2. Uploaded all chunks with embeddings
3. Verified the documents are indexed correctly

**Index structure:**
- 34 documents (chunks) indexed
- Each has text content + 3072-dim embedding
- Filterable by `content_type`
- Ready for semantic search!

### 📷 Verify in Azure Portal

**Next**: Let's search this index!

You can explore your indexed documents using the **Search Explorer** in the Azure Portal:

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Indexes → Search Explorer showing documents

![Search Explorer](images/search-index-documents.png)

---

# Part 3: Search Modes

## 🎯 Learning Goal
Understand the different search modes in Azure AI Search and when to use each one.

Azure AI Search supports multiple search modes, each with different strengths:

| Mode | How it Works | Best For | Weakness |
|------|-------------|----------|----------|
| **Text (BM25)** | Keyword matching + TF-IDF | Exact terms, "תחנה 36" | Misses synonyms |
| **Vector** | Cosine similarity on embeddings | Semantic meaning, multilingual | May miss exact matches |
| **Hybrid** | BM25 + Vector with RRF fusion | **General RAG** | Slightly more expensive |
| **Semantic** | Hybrid + L2 neural reranking | **Production RAG** | Requires Standard tier |

> 🎓 **Recommendation**: Use **Hybrid + Semantic** for production RAG systems. It combines the precision of keyword search with the recall of semantic search.

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Indexes → Semantic configurations

### 📷 Semantic Configuration in Portal

![Semantic Configuration](images/semantic-config.png)

Semantic ranking is configured in the index settings. Here's how it appears in the Azure Portal:

## Lab 3.1: Text Search (BM25)

**BM25** (Best Match 25) is a classic keyword-based ranking algorithm that:
- Counts word frequency (TF - Term Frequency)
- Penalizes common words (IDF - Inverse Document Frequency)
- Works great for exact term matches like "תחנה 36"

**Limitation**: Won't find "station" when searching for "תחנה" (different words, same meaning).

In [ ]:
# Cell 3.1: Text-only search (BM25)
from azure.search.documents.models import QueryType

def search_text(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform text-only (BM25) search.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    results = search_client.search(
        search_text=query,
        query_type=QueryType.SIMPLE,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test text search with Metro query
query = "נוסעים תחנה 36"  # "passengers station 36" in Hebrew
text_results = search_text(query, top=3)

print(f"🔍 Text Search (BM25): '{query}'\n")
for i, r in enumerate(text_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.2: Vector Search (Semantic)

**Vector search** finds documents by semantic similarity:
1. Convert query to embedding vector
2. Find k-nearest neighbors (kNN) in the vector space
3. Return documents with highest cosine similarity

**Superpower**: An English query can find Hebrew content (and vice versa) because both map to similar vectors!

**Limitation**: May miss exact keyword matches (searching for "36" might return "35" if semantically similar).

In [ ]:
# Cell 3.2: Vector-only search
from azure.search.documents.models import VectorizedQuery

def search_vector(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform vector-only (semantic) search.
    
    Args:
        query: Search query text (will be embedded)
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=top,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=None,  # No text search
        vector_queries=[vector_query],
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test vector search with natural language query (English!)
# Vector search can find Hebrew content from English query!
query = "How many passengers are expected at the station during peak hours?"
vector_results = search_vector(query, top=3)

print(f"🔍 Vector Search (Semantic): '{query}'\n")
print("💡 Notice: English query finds Hebrew content!\n")
for i, r in enumerate(vector_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.3: Hybrid Search (Text + Vector)

**Hybrid search** combines BM25 and vector search using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score = 1/(k + rank_bm25) + 1/(k + rank_vector)
```

This gives the best of both worlds:
- Exact keyword matching from BM25 ("תחנה 36" matches literally)
- Semantic understanding from vectors ("station" matches "תחנה")

In [ ]:
# Cell 3.3: Hybrid search (text + vector)

def search_hybrid(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform hybrid search (BM25 + vector with RRF fusion).
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,  # Over-fetch for RRF
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,        # Text search
        vector_queries=[vector_query],  # Vector search
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test hybrid search with mixed Hebrew/English query
query = "קיבולת נוסעים passengers capacity station 36"
hybrid_results = search_hybrid(query, top=3)

print(f"🔍 Hybrid Search (BM25 + Vector): '{query}'\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.4: Semantic Ranking (L2 Reranker)

### What is Semantic Ranking?

**Semantic ranking** (also called **L2 reranking**) is a two-stage retrieval process:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Stage 1 (L1): Hybrid Search                                            │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Query → BM25 + Vector → RRF Fusion → Top 50 candidates         │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                          │
│  Stage 2 (L2): Semantic Reranker                                        │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Transformer model scores each candidate for relevance          │   │
│  │  Returns reranker_score (0-4 scale) + final top K               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────┘
```

### Reranker Score (0-4 Scale)

| Score | Meaning |
|-------|---------|------------|
| 0 | Not relevant at all |
| 1 | Slightly relevant |
| 2 | Moderately relevant |
| 3 | Highly relevant |
| **4** | Perfect match |

> 💡 **Tip**: Filter results with `reranker_score >= 2` for quality answers.

In [ ]:
# Cell 3.4: Hybrid + Semantic ranking
from azure.search.documents.models import QueryType, QueryCaptionType, QueryAnswerType

def search_semantic(
    query: str, 
    top: int = 5, 
    filter: str = None,
    include_answers: bool = True
) -> dict:
    """
    Perform hybrid search with semantic ranking.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        include_answers: Extract semantic answers
        
    Returns:
        Dict with results and optional answers
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        query_answer=QueryAnswerType.EXTRACTIVE if include_answers else None,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    # Extract results
    output = {
        "results": [],
        "answers": []
    }
    
    # Get answers (if available)
    try:
        answers = results.get_answers()
        if answers:
            output["answers"] = [
                {
                    "text": a.text,
                    "score": a.score
                }
                for a in answers
            ]
    except:
        pass
    
    # Get documents
    for r in results:
        doc = {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"],
            "reranker_score": r.get("@search.reranker_score", None)
        }
        
        # Get captions
        captions = r.get("@search.captions", [])
        if captions:
            doc["caption"] = captions[0].text if hasattr(captions[0], "text") else str(captions[0])
        
        output["results"].append(doc)
    
    return output

# Test semantic search with Metro question
query = "כמה נוסעים צפויים בשעת שיא בתחנה 36?"  # "How many passengers expected at peak hour at Station 36?"
semantic_results = search_semantic(query, top=3)

print(f"🔍 Semantic Search: '{query}'\n")

# Show answers (if any)
if semantic_results["answers"]:
    print("📝 Extracted Answers:")
    for a in semantic_results["answers"]:
        print(f"   Score {a['score']:.2f}: {a['text']}")
    print()

# Show documents
print("📄 Documents:")
for i, r in enumerate(semantic_results["results"], 1):
    reranker_str = f", Reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}{reranker_str}")
    if r.get("caption"):
        print(f"   Caption: {r['caption'][:100]}...")
    print(f"   Content: {r['content'][:100]}...\n")

## Lab 3.5: Compare Search Modes

Let's compare all search modes side by side with Metro-specific queries:

In [ ]:
# Cell 3.5: Side-by-side comparison

def compare_search_modes(query: str, top: int = 3):
    """Compare different search modes for the same query."""
    print(f"="*80)
    print(f"Query: '{query}'")
    print(f"="*80)
    
    # Text search
    text_results = search_text(query, top)
    print(f"\n📖 TEXT (BM25):")
    for i, r in enumerate(text_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Vector search
    vector_results = search_vector(query, top)
    print(f"\n🧮 VECTOR (Cosine):")
    for i, r in enumerate(vector_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Hybrid search
    hybrid_results = search_hybrid(query, top)
    print(f"\n🔀 HYBRID (RRF):")
    for i, r in enumerate(hybrid_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Semantic search
    semantic_results = search_semantic(query, top, include_answers=False)
    print(f"\n🧠 SEMANTIC (L2 Reranker):")
    for i, r in enumerate(semantic_results["results"], 1):
        reranker = f", reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f}{reranker})")

# Test with different query types
print("\n" + "🔬 TEST 1: Hebrew exact term")
compare_search_modes("קיבולת נוסעים")  # "passenger capacity"

print("\n" + "🔬 TEST 2: English semantic question")
compare_search_modes("What is the expected number of passengers at the metro station?")

print("\n" + "🔬 TEST 3: Mixed language (Hebrew + English)")
compare_search_modes("מיקום התחנה location entrances")

### 🔬 Analyzing the Results

**What to observe:**
- **TEXT (BM25)**: Good for exact Hebrew terms, may miss English synonyms
- **VECTOR (Cosine)**: Finds semantically similar content across languages
- **HYBRID (RRF)**: Balances both - usually best overall ranking
- **SEMANTIC (L2)**: Reranker scores indicate true relevance (2+ is good)

**Notice**: Different search modes return different documents! The reranker score helps identify truly relevant results.

---

# Part 4: Retrieval Patterns for RAG

## 🎯 Learning Goal
Learn advanced retrieval strategies for different types of questions and content.

Different RAG scenarios require different retrieval strategies:

| Pattern | Use Case | Example Query |
|---------|----------|---------------|
| **Multi-Retriever** | Mixed content (text + tables + figures) | "Tell me about the station" |
| **Filtered Retrieval** | Content-type specific queries | "Show me the land use table" |
| **Intent-Aware** | Auto-detect what user wants | "What's in the diagram?" |

> 🎓 **Why this matters**: If you only do standard retrieval, tables and figures get drowned out by text chunks. Multi-retriever ensures each content type is represented.

## Lab 4.1: Multi-Retriever Pattern

**Problem**: Standard retrieval returns the top-K most similar documents overall. But if you have 30 text chunks and 4 table chunks, tables get drowned out!

**Solution**: Query each content type separately and merge results:
- Get top 2 text chunks
- Get top 2 table chunks  
- Get top 2 figure chunks
- Combine for a balanced context

This ensures tables (with land use data) and figures (with maps) appear in the context.

In [ ]:
# Cell 4.1: Multi-retriever with content-type awareness

def multi_retriever(
    query: str,
    top_per_type: int = 2,
    content_types: list[str] = ["text", "table", "figure"]
) -> dict:
    """
    Retrieve from each content type separately.
    
    This pattern ensures tables (land use data) and figures (maps)
    aren't drowned out by text chunks in the results.
    
    Args:
        query: Search query
        top_per_type: Results per content type
        content_types: Types to query
        
    Returns:
        Dict with results by content type
    """
    results = {}
    
    for ct in content_types:
        filter_expr = f"content_type eq '{ct}'"
        type_results = search_hybrid(query, top=top_per_type, filter=filter_expr)
        results[ct] = type_results
    
    return results

# Test multi-retriever with Metro query
query = "ייעודי קרקע land use"  # "land use" in Hebrew + English
multi_results = multi_retriever(query, top_per_type=2)

print(f"🔍 Multi-Retriever: '{query}'\n")

for content_type, results in multi_results.items():
    print(f"📁 {content_type.upper()} ({len(results)} results):")
    for r in results:
        print(f"   - {r['id']}: {r['content'][:80]}...")
    print()

## Lab 4.2: Filtered Retrieval

**Intent detection** narrows search based on what the user wants:
- "Show me the **table** of land use" → Filter to `content_type eq 'table'`
- "What does the **map** show?" → Filter to `content_type eq 'figure'`
- General questions → No filter

This is a simple keyword-based approach. In production, you'd use an LLM to detect intent.

In [ ]:
# Cell 4.2: Intent-based filtered retrieval

def detect_intent(query: str) -> str:
    """
    Simple intent detection for filtering.
    In production, use an LLM for this.
    """
    query_lower = query.lower()
    
    # Check for table indicators (land use, specifications, etc.)
    table_keywords = ["טבלה", "table", "ייעוד", "land use", "מפרט", "specifications", "נתונים", "data"]
    if any(kw in query_lower for kw in table_keywords):
        return "table"
    
    # Check for figure indicators (map, diagram, image, etc.)
    figure_keywords = ["מפה", "map", "תרשים", "diagram", "תמונה", "image", "figure", "show me", "הראה לי"]
    if any(kw in query_lower for kw in figure_keywords):
        return "figure"
    
    return "all"  # No specific intent


def intent_aware_search(query: str, top: int = 5) -> list[dict]:
    """
    Search with automatic intent detection.
    """
    intent = detect_intent(query)
    
    if intent == "table":
        filter_expr = "content_type eq 'table'"
        print(f"🎯 Detected intent: TABLE")
    elif intent == "figure":
        filter_expr = "content_type eq 'figure'"
        print(f"🎯 Detected intent: FIGURE")
    else:
        filter_expr = None
        print(f"🎯 Detected intent: GENERAL")
    
    return search_hybrid(query, top=top, filter=filter_expr)

# Test with different Metro queries
print("\n" + "="*50)
print("Query: 'מה ייעודי הקרקע באזור התחנה?'")
print("       (What are the land use types near the station?)")
results1 = intent_aware_search("מה ייעודי הקרקע באזור התחנה?")  # Land use query
print(f"Results: {[r['id'] for r in results1]}")

print("\n" + "="*50)
print("Query: 'Show me the map of station entrances'")
results2 = intent_aware_search("Show me the map of station entrances")
print(f"Results: {[r['id'] for r in results2]}")

print("\n" + "="*50)
print("Query: 'כמה כניסות יש לתחנה?'")
print("       (How many entrances does the station have?)")
results3 = intent_aware_search("כמה כניסות יש לתחנה?")  # General question
print(f"Results: {[r['id'] for r in results3]}")

## Lab 4.3: RAG Pipeline Integration

This cell creates the **retrieval** portion of a complete RAG pipeline:

```
User Question → Embed Query → Search Index → Top K Chunks → Format Context → Create Prompt
```

**Key design decisions:**
- Format tables and figures differently than text
- Include section headers for context
- Truncate long content to fit context window
- Support both Hebrew and English questions

In [ ]:
# Cell 4.3: Complete RAG query pipeline

def rag_query(
    question: str,
    top_k: int = 5,
    use_semantic: bool = True
) -> dict:
    """
    Complete RAG query: retrieve + format context for LLM.
    
    Args:
        question: User's question
        top_k: Number of chunks to retrieve
        use_semantic: Use semantic ranking
        
    Returns:
        Dict with retrieved context and metadata
    """
    # Step 1: Retrieve relevant chunks
    if use_semantic:
        search_result = search_semantic(question, top=top_k)
        chunks = search_result["results"]
    else:
        chunks = search_hybrid(question, top=top_k)
    
    # Step 2: Format context for LLM
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        # Format based on content type
        if chunk["content_type"] == "table":
            context_parts.append(f"[Table {i}]:\n{chunk['content']}")
        elif chunk["content_type"] == "figure":
            context_parts.append(f"[Figure {i} description]:\n{chunk['content']}")
        else:
            section = chunk.get('section_header', '')
            section_label = f" (Section: {section})" if section else ""
            context_parts.append(f"[Text {i}{section_label}]:\n{chunk['content']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Create prompt template (supports Hebrew + English)
    prompt = f"""Based on the following context about Metro Station 36 (תחנה 36 - שדרות הציונות), answer the question.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:"""
    
    return {
        "question": question,
        "retrieved_chunks": len(chunks),
        "content_types": [c["content_type"] for c in chunks],
        "context_length": len(context),
        "prompt": prompt,
        "chunks": chunks
    }

# Test RAG query with Metro question
question = "כמה נוסעים צפויים בתחנה 36 בשעת השיא?"  # "How many passengers expected at Station 36 during peak hour?"
rag_result = rag_query(question, top_k=3)

print(f"🤖 RAG Query: '{question}'\n")
print(f"📊 Retrieved: {rag_result['retrieved_chunks']} chunks")
print(f"📁 Content types: {rag_result['content_types']}")
print(f"📏 Context length: {rag_result['context_length']} chars")
print(f"\n{'='*60}")
print("PROMPT (truncated):")
print(f"{'='*60}")
print(rag_result['prompt'][:1500] + "...")

## Lab 4.4: Generate Answer with GPT-4.1

Now we connect retrieval to generation - the complete **RAG pipeline**:

```
User Question → Retrieve Chunks → Format Context → LLM → Answer
```

This is where everything comes together! The quality of the answer depends on:
1. **Retrieval quality**: Did we find the right chunks?
2. **Context formatting**: Is the context clear for the LLM?
3. **LLM capability**: Can the model reason over the context?

In [ ]:
# Cell 4.4: Complete RAG with answer generation

def ask(question: str, top_k: int = 5) -> str:
    """
    Full RAG pipeline: retrieve → generate answer.
    
    Args:
        question: User's question (Hebrew or English)
        top_k: Number of chunks to retrieve
        
    Returns:
        Generated answer
    """
    # Retrieve context
    rag_result = rag_query(question, top_k=top_k)
    
    # Generate answer
    chat_model = env.get("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1")
    
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": """You are a helpful assistant for Metro Station 36 (תחנה 36 - שדרות הציונות) in Rishon LeZion, Israel.
Answer questions based on the provided context. You can respond in Hebrew or English, matching the language of the question.
If the context doesn't contain enough information, say so."""
            },
            {
                "role": "user",
                "content": rag_result["prompt"]
            }
        ],
        temperature=0.3,
        max_tokens=500
    )
    
    return response.choices[0].message.content

# Test full RAG pipeline with Metro questions
# Using top_k=5 to get more context (passenger data might be in different chunks)
question = "איפה נמצאת תחנה 36?"  # "Where is Station 36 located?"
print(f"❓ Question: {question}")
print("   (Where is Station 36 located?)")
print("\n⏳ Generating answer...\n")

answer = ask(question, top_k=5)
print(f"💡 Answer:\n{answer}")

In [ ]:
# Cell 4.5: Test more Metro questions
# 
# Let's test the RAG pipeline with different types of questions.
# Notice how some questions work better than others - this reveals
# the importance of good chunking and retrieval strategy!

test_questions = [
    ("Where is Station 36 located?", "English question about location"),
    ("מה ייעודי הקרקע באזור התחנה?", "Hebrew question about land use"),
    ("What types of land use are planned near the station?", "English question about land use"),
]

for q, description in test_questions:
    print(f"\n{'='*60}")
    print(f"❓ {q}")
    print(f"   ({description})")
    print(f"{'='*60}")
    answer = ask(q, top_k=5)  # Use 5 chunks for better context
    print(f"\n💡 {answer}")

---

# Part 5: Agentic Retrieval (Preview)

## 🎯 Learning Goal
Understand Agentic Retrieval for complex multi-part questions (advanced feature).

> ⚠️ **IMPORTANT**: Agentic Retrieval requires **Standard tier (S1) or higher** Azure AI Search.  
> If you're on Basic tier, the cells below will be **skipped automatically** - this is expected behavior, not an error!

## What is Agentic Retrieval?

**Agentic Retrieval** is a multi-query pipeline in Azure AI Search designed for complex questions. It uses an LLM to decompose your question into focused subqueries.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     Traditional RAG vs Agentic Retrieval                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  TRADITIONAL RAG:                                                           │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question → Single Query → Search → Top K → LLM → Answer   │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
│  AGENTIC RETRIEVAL:                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question                                                   │       │
│  │         ↓                                                        │       │
│  │  LLM Query Planning (decompose into focused subqueries)          │       │
│  │         ↓                                                        │       │
│  │  ┌─────────┐  ┌─────────┐  ┌─────────┐                          │       │
│  │  │Subquery1│  │Subquery2│  │Subquery3│  (parallel execution)    │       │
│  │  └────┬────┘  └────┬────┘  └────┬────┘                          │       │
│  │       ↓            ↓            ↓                                │       │
│  │  Semantic Rerank + Merge Results                                 │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 🔑 Key Difference: Agentic Index vs Normal Index

Our normal index (`module5-metro-index`) has **pre-computed embeddings** - we embed text during upload.

An **agentic-ready index** needs an **integrated vectorizer** that embeds queries at search time:

```
┌──────────────────────────────────────────────────────────────────┐
│  Normal Index (Parts 1-4)           │  Agentic Index (Part 5)    │
├─────────────────────────────────────┼────────────────────────────┤
│  • Pre-computed embeddings          │  • Pre-computed embeddings │
│  • Manual query embedding           │  • Integrated VECTORIZER   │
│  • Semantic config                  │  • Default semantic config │
│  • Standard search APIs             │  • Knowledge Source/Base   │
└─────────────────────────────────────┴────────────────────────────┘
```

**Why a vectorizer?** Agentic retrieval generates subqueries dynamically. The vectorizer converts these text subqueries to vectors automatically - no client-side embedding needed!

### Service Tier Requirements

| Tier | Agentic Retrieval | What You Can Do |
|------|-------------------|-----------------|
| Free | ❌ | Basic search only |
| Basic | ❌ | Parts 1-4 of this lab |
| **Standard (S1+)** | ✅ | Full lab including Part 5 |

### 📷 Checking Your Service Tier

To verify your Azure AI Search tier, go to the Azure Portal:

![Service Tier](images/search-tier-upgrade.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Settings → Scale showing tier selection

## Lab 5.0: Prerequisites Check

The cell below checks if your Azure AI Search service supports Agentic Retrieval.

**Expected outcomes:**
- ✅ **Standard tier**: You'll see "Prerequisites met!" and can try Parts 5.1-5.4
- ⚠️ **Basic tier**: Cells will be skipped - this is expected!

### Architecture Overview

We'll create these components:
```
┌─────────────────────────────────────────────────────────────────────┐
│  Agentic Retrieval Architecture                                      │
│                                                                      │
│  ┌──────────────────────────────────────────────────────────────┐   │
│  │  Agentic Index: module5-metro-agentic                        │   │
│  │  ├─ Vector field with VECTORIZER (auto-embeds queries)       │   │
│  │  ├─ Default semantic configuration                            │   │
│  │  └─ Index description (helps agent choose index)             │   │
│  └──────────────────────────────────────────────────────────────┘   │
│                              ↓                                       │
│  ┌──────────────────────────────────────────────────────────────┐   │
│  │  Knowledge Source: module5-metro-ks                          │   │
│  │  └─ Points to agentic index, specifies semantic config       │   │
│  └──────────────────────────────────────────────────────────────┘   │
│                              ↓                                       │
│  ┌──────────────────────────────────────────────────────────────┐   │
│  │  Knowledge Base: module5-metro-kb                            │   │
│  │  ├─ References knowledge source(s)                           │   │
│  │  └─ Connects to LLM for query planning                       │   │
│  └──────────────────────────────────────────────────────────────┘   │
│                              ↓                                       │
│  ┌──────────────────────────────────────────────────────────────┐   │
│  │  Retrieve API: Query with automatic decomposition            │   │
│  └──────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────┘
```

## 🔐 Lab 5.0a: Configure RBAC for Agentic Retrieval

**Agentic Retrieval requires specific Azure RBAC roles AND authentication mode.** Run the cell below to automatically configure:

1. **Enable Managed Identity** on your Azure AI Search service
2. **Assign Search roles** to your user account
3. **Grant Search→OpenAI access** for the knowledge base to call Azure OpenAI
4. **Enable RBAC data plane authentication** (critical for Entra ID auth to work)

> ⚠️ **You must be logged in with Azure CLI** (`az login`) and have **Owner** or **User Access Administrator** role on your subscription to assign roles.

### What Roles Are Required?

| Role | Assigned To | Purpose |
|------|-------------|---------|
| Search Service Contributor | Your user | Manage search indexes, knowledge sources/bases |
| Search Index Data Contributor | Your user | Write data to search indexes |
| Search Index Data Reader | Your user | Read data from search indexes |
| Cognitive Services OpenAI User | Search Service (MSI) | Allow search to call Azure OpenAI |

### Why Enable RBAC Data Plane Auth?

By default, Azure AI Search uses API keys for data plane operations. To use Entra ID (RBAC) authentication for retrieval operations, you must explicitly enable the `aadOrApiKey` auth mode with `http401WithBearerChallenge`. This is automatically configured in the cell below.

### 📷 RBAC Configuration in Azure Portal

Here's what the RBAC settings look like in the Azure Portal:

**Managed Identity** (required for Search→OpenAI communication):

![Managed Identity](images/search-managed-identity.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Identity → System Assigned = ON

**Authentication Options** (API Keys and/or RBAC):

![Auth Options](images/search-auth-options.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Settings → Keys showing "Both" option

**Role Assignments** (RBAC roles on the Search service):

![Role Assignments](images/search-rbac-roles.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Access control (IAM) → Role assignments

In [ ]:
# Cell 5.0a: Configure RBAC for Agentic Retrieval
#
# This cell configures the required Azure RBAC roles for Agentic Retrieval.
# You only need to run this ONCE per Azure environment.
#
# Requirements:
# - Azure CLI installed and logged in (`az login`)
# - Owner or User Access Administrator role on your subscription

import subprocess
import json

print("🔐 Configuring RBAC for Agentic Retrieval...\n")

# Get resource information from environment
SEARCH_SERVICE_NAME = env["AZURE_SEARCH_ENDPOINT"].replace("https://", "").split(".")[0]
OPENAI_ENDPOINT = env["AZURE_OPENAI_ENDPOINT"]
OPENAI_RESOURCE_NAME = OPENAI_ENDPOINT.replace("https://", "").split(".")[0]

print(f"📦 Resources:")
print(f"   Search Service: {SEARCH_SERVICE_NAME}")
print(f"   OpenAI Resource: {OPENAI_RESOURCE_NAME}")

# Step 1: Get current user's Object ID
print("\n1️⃣ Getting current user info...")
USER_OBJECT_ID = None
try:
    result = subprocess.run(
        'az ad signed-in-user show --query "{objectId: id, userPrincipalName: userPrincipalName}" -o json',
        shell=True, capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        user_info = json.loads(result.stdout)
        USER_OBJECT_ID = user_info.get('objectId')
        USER_NAME = user_info.get('userPrincipalName')
        print(f"   ✅ User: {USER_NAME}")
        print(f"   ✅ Object ID: {USER_OBJECT_ID}")
    else:
        print(f"   ❌ Failed to get user info: {result.stderr}")
        print("   Make sure you're logged in: az login")
except Exception as e:
    print(f"   ❌ Error: {e}")

# Step 2: Find Search Service Resource ID using az resource list (more reliable)
print("\n2️⃣ Finding Search Service Resource...")
SEARCH_RESOURCE_ID = None
SEARCH_MSI_ID = None
try:
    # Use az resource list which is more reliable across resource types
    cmd = f'az resource list --name "{SEARCH_SERVICE_NAME}" --resource-type "Microsoft.Search/searchServices" --query "[0].id" -o tsv'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    if result.returncode == 0 and result.stdout.strip():
        SEARCH_RESOURCE_ID = result.stdout.strip()
        print(f"   ✅ Search Resource ID found")
    else:
        # Fallback: try to find by name pattern
        print("   ⚠️ Direct lookup failed, trying subscription scan...")
        cmd = f'az resource list --query "[?contains(name, \'{SEARCH_SERVICE_NAME}\') && type==\'Microsoft.Search/searchServices\'].id" -o tsv'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        if result.returncode == 0 and result.stdout.strip():
            SEARCH_RESOURCE_ID = result.stdout.strip().split('\n')[0]
            print(f"   ✅ Search Resource ID found via scan")
        else:
            print(f"   ❌ Could not find Search service")
except Exception as e:
    print(f"   ❌ Error finding search service: {e}")

# Step 2b: Enable Managed Identity
if SEARCH_RESOURCE_ID:
    print("\n   Enabling Managed Identity...")
    try:
        # Enable system-assigned managed identity using az resource update
        cmd = f'az resource update --ids "{SEARCH_RESOURCE_ID}" --set identity.type=SystemAssigned --query "identity.principalId" -o tsv 2>/dev/null'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        if result.returncode == 0 and result.stdout.strip():
            SEARCH_MSI_ID = result.stdout.strip()
            print(f"   ✅ Managed Identity enabled/verified")
            print(f"   ✅ Search MSI Principal ID: {SEARCH_MSI_ID}")
        else:
            # Try to get existing MSI
            cmd2 = f'az resource show --ids "{SEARCH_RESOURCE_ID}" --query "identity.principalId" -o tsv 2>/dev/null'
            result2 = subprocess.run(cmd2, shell=True, capture_output=True, text=True, timeout=30)
            if result2.returncode == 0 and result2.stdout.strip():
                SEARCH_MSI_ID = result2.stdout.strip()
                print(f"   ✅ Managed Identity already enabled")
                print(f"   ✅ Search MSI Principal ID: {SEARCH_MSI_ID}")
            else:
                print(f"   ⚠️ Could not enable/verify Managed Identity automatically")
                print(f"      → Go to Azure Portal → AI Search → Identity → Enable System Assigned")
    except Exception as e:
        print(f"   ❌ Error enabling MSI: {e}")

# Step 3: Get OpenAI Resource ID
print("\n3️⃣ Getting OpenAI Resource info...")
OPENAI_RESOURCE_ID = None
try:
    cmd = f'az resource list --name "{OPENAI_RESOURCE_NAME}" --resource-type "Microsoft.CognitiveServices/accounts" --query "[0].id" -o tsv'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    if result.returncode == 0 and result.stdout.strip():
        OPENAI_RESOURCE_ID = result.stdout.strip()
        print(f"   ✅ OpenAI Resource ID found")
    else:
        # Try broader search - get first part of name for partial match
        name_prefix = OPENAI_RESOURCE_NAME.split("-")[0] if "-" in OPENAI_RESOURCE_NAME else OPENAI_RESOURCE_NAME[:10]
        cmd = f"az cognitiveservices account list --query \"[?contains(name, '{name_prefix}')].id\" -o tsv"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
        if result.returncode == 0 and result.stdout.strip():
            OPENAI_RESOURCE_ID = result.stdout.strip().split('\n')[0]
            print(f"   ✅ OpenAI Resource ID found via search")
        else:
            print(f"   ⚠️ Could not find OpenAI resource")
except Exception as e:
    print(f"   ❌ Error: {e}")

# Step 4: Assign roles to user on Search Service
if USER_OBJECT_ID and SEARCH_RESOURCE_ID:
    print("\n4️⃣ Assigning Search roles to your user...")
    
    search_roles = [
        ("Search Service Contributor", "7ca78c08-252a-4471-8644-bb5ff32d4ba0"),
        ("Search Index Data Contributor", "8ebe5a00-799e-43f5-93ac-243d3dce84a7"),
        ("Search Index Data Reader", "1407120a-92aa-4202-b7e9-c0e197c71c8f"),
    ]
    
    for role_name, role_id in search_roles:
        try:
            cmd = f'az role assignment create --assignee "{USER_OBJECT_ID}" --role "{role_id}" --scope "{SEARCH_RESOURCE_ID}" -o none 2>&1'
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
            output = (result.stdout + result.stderr).lower()
            if "already exists" in output or result.returncode == 0:
                print(f"   ✅ {role_name}")
            else:
                error_msg = (result.stderr or result.stdout)[:60] if (result.stderr or result.stdout) else 'unknown error'
                print(f"   ⚠️ {role_name}: {error_msg}")
        except Exception as e:
            print(f"   ❌ {role_name}: {e}")
else:
    print("\n4️⃣ ⚠️ Skipping user role assignment (missing user ID or Search resource)")

# Step 5: Assign Cognitive Services OpenAI User role to Search MSI
if SEARCH_MSI_ID and OPENAI_RESOURCE_ID:
    print("\n5️⃣ Granting Search service access to Azure OpenAI...")
    try:
        # Cognitive Services OpenAI User role
        cmd = f'az role assignment create --assignee "{SEARCH_MSI_ID}" --role "5e0bd9bd-7b93-4f28-af87-19fc36ad61bd" --scope "{OPENAI_RESOURCE_ID}" -o none 2>&1'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
        output = (result.stdout + result.stderr).lower()
        if "already exists" in output or result.returncode == 0:
            print(f"   ✅ Cognitive Services OpenAI User role assigned to Search MSI")
        else:
            error_msg = (result.stderr or result.stdout)[:80] if (result.stderr or result.stdout) else 'unknown error'
            print(f"   ⚠️ Could not assign role: {error_msg}")
    except Exception as e:
        print(f"   ❌ Error: {e}")
elif not SEARCH_MSI_ID:
    print("\n5️⃣ ⚠️ Skipping OpenAI role assignment (Search MSI not available)")
    print("      → Enable Managed Identity first in Azure Portal")
else:
    print("\n5️⃣ ⚠️ Skipping OpenAI role assignment (OpenAI resource not found)")

# Step 6: Enable RBAC Data Plane Authentication on Search Service
# This is CRITICAL - without this, RBAC roles won't work for data plane operations
if SEARCH_SERVICE_NAME:
    print("\n6️⃣ Enabling RBAC data plane authentication on Search service...")
    try:
        # Get resource group from resource ID
        rg_name = None
        if SEARCH_RESOURCE_ID:
            parts = SEARCH_RESOURCE_ID.split("/")
            rg_idx = parts.index("resourceGroups") if "resourceGroups" in parts else -1
            if rg_idx >= 0 and rg_idx + 1 < len(parts):
                rg_name = parts[rg_idx + 1]
        
        if rg_name:
            # Enable aadOrApiKey auth mode with bearer challenge for proper RBAC
            cmd = f'az search service update --name "{SEARCH_SERVICE_NAME}" --resource-group "{rg_name}" --auth-options aadOrApiKey --aad-auth-failure-mode http401WithBearerChallenge -o none 2>&1'
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
            output = (result.stdout + result.stderr).lower()
            if result.returncode == 0:
                print(f"   ✅ RBAC data plane authentication enabled")
                print(f"      Mode: aadOrApiKey with http401WithBearerChallenge")
            else:
                error_msg = (result.stderr or result.stdout)[:80] if (result.stderr or result.stdout) else 'unknown error'
                print(f"   ⚠️ Could not enable RBAC auth: {error_msg}")
        else:
            print(f"   ⚠️ Could not determine resource group from resource ID")
    except Exception as e:
        print(f"   ❌ Error: {e}")
else:
    print("\n6️⃣ ⚠️ Skipping RBAC auth mode config (Search service name not found)")

# Summary
print("\n" + "="*60)
print("📋 RBAC Configuration Summary")
print("="*60)

all_configured = USER_OBJECT_ID and SEARCH_RESOURCE_ID and SEARCH_MSI_ID and OPENAI_RESOURCE_ID

if all_configured:
    print("✅ All RBAC roles and authentication configured successfully!")
    print("\n   Your environment is ready for Agentic Retrieval.")
    print("   Wait 1-2 minutes for role propagation, then proceed.")
    RBAC_CONFIGURED = True
else:
    print("⚠️ Some RBAC configuration may need manual steps:")
    if not SEARCH_RESOURCE_ID:
        print("   ❌ Search service not found - check Azure CLI subscription")
    if not SEARCH_MSI_ID:
        print("   ❌ Search MSI not enabled - do this in Azure Portal:")
        print("      → AI Search → Identity → System Assigned → ON")
    if not OPENAI_RESOURCE_ID:
        print("   ❌ OpenAI resource not found - check resource names")
    print("\n   After manual fixes, re-run this cell.")
    RBAC_CONFIGURED = False

print("\n💡 Tip: Role propagation can take 1-2 minutes. If retrieval still fails,")
print("   wait a moment and try again.")

In [ ]:
# Cell 5.0: Check prerequisites for Agentic Retrieval
import subprocess

print("🔍 Checking Agentic Retrieval Prerequisites...\n")

# Extract service name from endpoint
SEARCH_SERVICE_NAME = SEARCH_ENDPOINT.replace("https://", "").split(".")[0]
print(f"   Search Service: {SEARCH_SERVICE_NAME}")

# Try to get service info via Azure CLI
try:
    result = subprocess.run(
        f'az search service show --name "{SEARCH_SERVICE_NAME}" --query "{{sku: sku.name, semanticSearch: semanticSearch}}" -o json 2>/dev/null',
        shell=True, capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0 and result.stdout.strip():
        import json
        info = json.loads(result.stdout)
        sku = info.get('sku', 'unknown')
        semantic = info.get('semanticSearch', 'disabled')
        print(f"   SKU: {sku}")
        print(f"   Semantic Search: {semantic}")
        
        if sku.lower() in ['basic', 'free']:
            print("\n⚠️  WARNING: Agentic Retrieval requires Standard tier or higher.")
            print("   Your service is on Basic/Free tier. Labs 5.1+ will be skipped.")
            print("   To upgrade: Azure Portal → Search Service → Settings → Pricing tier")
            AGENTIC_AVAILABLE = False
        elif semantic == 'disabled' or semantic is None:
            print("\n⚠️  WARNING: Semantic search is not enabled.")
            print("   Enable it: Azure Portal → Search Service → Settings → Premium features")
            AGENTIC_AVAILABLE = False
        else:
            print("\n✅ Prerequisites met! Agentic Retrieval is available.")
            AGENTIC_AVAILABLE = True
    else:
        print("   (Could not verify via Azure CLI - will try anyway)")
        AGENTIC_AVAILABLE = True
except Exception as e:
    print(f"   (Azure CLI check failed: {e})")
    print("   Will attempt Agentic Retrieval - it may fail if prerequisites aren't met.")
    AGENTIC_AVAILABLE = True

# Define agentic index name (separate from our normal index!)
AGENTIC_INDEX_NAME = "module5-metro-agentic"
KNOWLEDGE_SOURCE_NAME = "module5-metro-ks"
KNOWLEDGE_BASE_NAME = "module5-metro-kb"

print(f"\n📦 Agentic Resources to create:")
print(f"   - Agentic Index: {AGENTIC_INDEX_NAME}")
print(f"   - Knowledge Source: {KNOWLEDGE_SOURCE_NAME}")
print(f"   - Knowledge Base: {KNOWLEDGE_BASE_NAME}")

In [ ]:
# Cell 5.1: Install preview SDK and create AGENTIC INDEX with Vectorizer
#
# This is the KEY DIFFERENCE from our normal index:
# - Normal index: We embed text during upload, query embedding is done client-side
# - Agentic index: Has an integrated VECTORIZER that embeds queries at search time
#
# The vectorizer is REQUIRED for agentic retrieval because the system generates
# subqueries dynamically and needs to embed them automatically.

if AGENTIC_AVAILABLE:
    import sys
    print("📦 Installing PREVIEW SDK for Agentic Retrieval...")
    print("   (This includes classes like SearchIndexKnowledgeSource, KnowledgeBase...)")
    
    # Use --pre to get the LATEST preview with agentic retrieval support
    # The Microsoft docs require: pip install azure-identity requests azure-search-documents --pre
    !{sys.executable} -m pip install -q --upgrade "azure-search-documents" --pre
    
    # Verify version
    import importlib.metadata
    version = importlib.metadata.version('azure-search-documents')
    print(f"✅ Installed azure-search-documents version: {version}")
    
    # Check if version is preview (should be 11.6.0bX or higher)
    if 'b' not in version and int(version.split('.')[1]) < 7:
        print("⚠️  Warning: Version may not include agentic retrieval APIs.")
        print("   Try: pip install azure-search-documents --pre --force-reinstall")
    
    # Need to reload modules after upgrade
    print("\n⚠️  IMPORTANT: Restart the kernel after running this cell!")
    print("   Menu → Kernel → Restart Kernel")
    print("   Then re-run cells from 0.1 onwards.\n")
    
    # Import agentic-specific models for index creation
    try:
        from azure.search.documents.indexes.models import (
            SearchIndex,
            SearchField,
            SearchFieldDataType,
            SearchableField,
            SimpleField,
            VectorSearch,
            HnswAlgorithmConfiguration,
            VectorSearchProfile,
            SemanticConfiguration,
            SemanticField,
            SemanticPrioritizedFields,
            SemanticSearch,
            # NEW for Agentic: Vectorizer!
            AzureOpenAIVectorizer,
            AzureOpenAIVectorizerParameters,
        )
        print("✅ Vectorizer classes imported successfully")
    except ImportError as e:
        print(f"⚠️  Could not import vectorizer classes: {e}")
        print("   Please restart kernel and try again.")
        AGENTIC_INDEX_CREATED = False
        raise
    
    print("\n🔧 Creating Agentic Index with Vectorizer...")
    
    # Get Azure OpenAI details for the vectorizer
    AZURE_OPENAI_ENDPOINT = env["AZURE_OPENAI_ENDPOINT"]
    EMBEDDING_DEPLOYMENT = env.get("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")
    
    # The vectorizer will call Azure OpenAI at query time to embed subqueries
    vectorizer = AzureOpenAIVectorizer(
        vectorizer_name="aoai-vectorizer",
        parameters=AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT,
            deployment_name=EMBEDDING_DEPLOYMENT,
            model_name="text-embedding-3-large",
        )
    )
    
    # Vector search config with vectorizer (the key difference!)
    agentic_vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw-config",
                parameters={
                    "m": 4,
                    "efConstruction": 400,
                    "efSearch": 500,
                    "metric": "cosine"
                }
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-config",
                vectorizer_name="aoai-vectorizer"  # Link to vectorizer!
            )
        ],
        vectorizers=[vectorizer]  # The vectorizer definition
    )
    
    # Semantic config with DEFAULT configuration (required for agentic)
    agentic_semantic_config = SemanticConfiguration(
        name="semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            content_fields=[SemanticField(field_name="content")],
        )
    )
    
    agentic_semantic_search = SemanticSearch(
        default_configuration_name="semantic-config",  # Set as DEFAULT
        configurations=[agentic_semantic_config]
    )
    
    # Same fields as our normal index
    agentic_fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String, searchable=True, 
                       analyzer_name="standard.lucene"),
        SimpleField(name="content_type", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchableField(name="section_header", type=SearchFieldDataType.String, searchable=True, filterable=True),
        SimpleField(name="strategy", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=EMBEDDING_DIMENSIONS,
            vector_search_profile_name="vector-profile"
        ),
        SimpleField(name="metadata", type=SearchFieldDataType.String, filterable=False),
    ]
    
    # Create the agentic index with description (helps agent decide to use it)
    agentic_index = SearchIndex(
        name=AGENTIC_INDEX_NAME,
        description="Metro Station 36 (תחנה 36 - שדרות הציונות) technical documents including station specifications, passenger forecasts, land use data, and location maps. Contains Hebrew and English content.",
        fields=agentic_fields,
        vector_search=agentic_vector_search,
        semantic_search=agentic_semantic_search
    )
    
    # Create or update index
    try:
        result = index_client.create_or_update_index(agentic_index)
        print(f"✅ Agentic Index '{result.name}' created")
        print(f"   - Vectorizer: {vectorizer.vectorizer_name} (auto-embeds queries)")
        print(f"   - Default semantic config: semantic-config")
        print(f"   - Description: {result.description[:60]}...")
        AGENTIC_INDEX_CREATED = True
    except Exception as e:
        print(f"❌ Error creating agentic index: {e}")
        AGENTIC_INDEX_CREATED = False
else:
    print("⏭️  Skipping - Agentic Retrieval not available on this service tier.")
    AGENTIC_INDEX_CREATED = False

In [ ]:
# Cell 5.2: Upload documents to Agentic Index
#
# We reuse our enriched_chunks from Part 1 (they already have embeddings).
# The agentic index stores the same data, but has a vectorizer for query-time embedding.

if AGENTIC_AVAILABLE and AGENTIC_INDEX_CREATED:
    from azure.search.documents import SearchClient
    
    # Create search client for the agentic index
    agentic_search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=AGENTIC_INDEX_NAME,
        credential=search_credential
    )
    
    # Prepare documents (same as Part 2)
    agentic_documents = [
        {
            "id": chunk["id"],
            "content": chunk["content"],
            "content_type": chunk.get("content_type", "text"),
            "section_header": chunk.get("section_header", ""),
            "strategy": chunk.get("strategy", "unknown"),
            "embedding": chunk["embedding"],
            "metadata": json.dumps(chunk.get("metadata", {}))
        }
        for chunk in enriched_chunks
    ]
    
    print(f"📤 Uploading {len(agentic_documents)} documents to Agentic Index...")
    
    # Upload in batches
    batch_size = 100
    total_uploaded = 0
    
    for i in range(0, len(agentic_documents), batch_size):
        batch = agentic_documents[i:i + batch_size]
        try:
            result = agentic_search_client.upload_documents(documents=batch)
            succeeded = sum(1 for r in result if r.succeeded)
            total_uploaded += succeeded
        except Exception as e:
            print(f"   Batch error: {e}")
    
    print(f"✅ Uploaded {total_uploaded} documents to '{AGENTIC_INDEX_NAME}'")
    
    # Wait for index to update
    import time
    time.sleep(2)
    
    # Verify
    verify_results = agentic_search_client.search(search_text="*", include_total_count=True)
    doc_count = verify_results.get_count()
    print(f"   Index now contains {doc_count} documents")
    AGENTIC_DOCS_UPLOADED = True
else:
    print("⏭️  Skipping - Agentic index not created.")
    AGENTIC_DOCS_UPLOADED = False

In [ ]:
# Cell 5.3: Create Knowledge Source and Knowledge Base
#
# Knowledge Source: Wraps an index and specifies which fields to use
# Knowledge Base: Groups knowledge sources and connects to LLM for query planning
#
# CORRECT IMPORTS based on Microsoft docs (2025-11-01-preview API):
# https://learn.microsoft.com/en-us/azure/search/search-get-started-agentic-retrieval

if AGENTIC_AVAILABLE and AGENTIC_DOCS_UPLOADED:
    import sys
    import importlib
    
    # Force complete reload of azure.search.documents hierarchy
    print("🔄 Forcing module reload to pick up new SDK classes...")
    
    # Remove all cached azure modules
    azure_mods = [k for k in sys.modules.keys() if k.startswith('azure.search')]
    for mod in azure_mods:
        del sys.modules[mod]
    print(f"   Cleared {len(azure_mods)} cached modules")
    
    # Now import fresh
    from azure.search.documents.indexes import models as idx_models
    knowledge_exports = [x for x in dir(idx_models) if 'Knowledge' in x and not x.startswith('_')]
    print(f"   Available Knowledge classes after reload: {len(knowledge_exports)}")
    
    try:
        # These imports are from azure.search.documents.indexes.models
        # Available in azure-search-documents >= 11.7.0b2
        from azure.search.documents.indexes.models import (
            SearchIndexKnowledgeSource,
            SearchIndexKnowledgeSourceParameters,
            SearchIndexFieldReference,
            KnowledgeBase,
            KnowledgeBaseAzureOpenAIModel,
            AzureOpenAIVectorizerParameters,
            KnowledgeSourceReference,
            KnowledgeRetrievalOutputMode,
        )
        
        print("✅ Agentic retrieval classes imported successfully!")
        print("🔧 Creating Knowledge Source...")
        
        # Need to recreate the index_client since we cleared modules
        from azure.search.documents.indexes import SearchIndexClient
        from azure.core.credentials import AzureKeyCredential
        
        index_client = SearchIndexClient(
            endpoint=SEARCH_ENDPOINT,
            credential=AzureKeyCredential(SEARCH_API_KEY)
        )
        
        # Knowledge Source - wraps our agentic index
        knowledge_source = SearchIndexKnowledgeSource(
            name=KNOWLEDGE_SOURCE_NAME,
            description="Metro Station 36 documents for RAG workshop - includes specs, forecasts, land use, maps",
            search_index_parameters=SearchIndexKnowledgeSourceParameters(
                search_index_name=AGENTIC_INDEX_NAME,
                # Fields to include in citations/references (human-readable only)
                source_data_fields=[
                    SearchIndexFieldReference(name="content_type"),
                    SearchIndexFieldReference(name="section_header"),
                ],
            )
        )
        
        result = index_client.create_or_update_knowledge_source(knowledge_source)
        print(f"✅ Knowledge Source '{result.name}' created")
        print(f"   - Points to index: {AGENTIC_INDEX_NAME}")
        
        # Knowledge Base - groups knowledge sources and connects to LLM
        print("\n🔧 Creating Knowledge Base...")
        
        GPT_MODEL = env.get("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1")
        
        # Azure OpenAI parameters for the LLM that does query planning
        aoai_params = AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT,
            deployment_name=GPT_MODEL,
            model_name=GPT_MODEL,
        )
        
        knowledge_base = KnowledgeBase(
            name=KNOWLEDGE_BASE_NAME,
            description="Knowledge base for Metro Station 36 documents",
            knowledge_sources=[
                KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)
            ],
            models=[
                KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)
            ],
            output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
            answer_instructions="Provide a concise answer based on the retrieved Metro Station 36 documents."
        )
        
        result = index_client.create_or_update_knowledge_base(knowledge_base)
        print(f"✅ Knowledge Base '{result.name}' created")
        print(f"   - Knowledge Source: {KNOWLEDGE_SOURCE_NAME}")
        print(f"   - LLM for query planning: {GPT_MODEL}")
        print(f"   - Output mode: ANSWER_SYNTHESIS")
        
        KNOWLEDGE_BASE_CREATED = True
        
    except ImportError as e:
        print(f"\n⚠️  Import error after reload: {e}")
        print("   The kernel may have deeply cached the old module structure.")
        print("\n   To fix completely:")
        print("   1. Close VS Code completely")
        print("   2. Reopen the project and notebook")
        print("   3. Run all cells again")
        KNOWLEDGE_BASE_CREATED = False
    except Exception as e:
        print(f"❌ Error creating knowledge base: {e}")
        KNOWLEDGE_BASE_CREATED = False
else:
    if not AGENTIC_AVAILABLE:
        print("⏭️  Skipping - Agentic Retrieval not available on this service tier.")
    else:
        print("⏭️  Skipping - Documents not uploaded to agentic index.")
    KNOWLEDGE_BASE_CREATED = False

# Note: After running this cell, you can see Knowledge Sources and Knowledge Bases 
# in Azure Portal under your AI Search service (Preview features).

### 📷 Knowledge Sources and Bases in Azure Portal

After creating the Knowledge Source and Knowledge Base, you can view them in the Azure Portal:

**Knowledge Sources** (maps your index to the retrieval system):

![Knowledge Sources](images/knowledge-sources.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Knowledge sources (Preview)

**Knowledge Bases** (connects sources to LLM for query planning):

![Knowledge Bases](images/knowledge-bases.png)

> 📸 **TODO**: Add screenshot of Azure Portal → AI Search → Knowledge bases (Preview)

In [ ]:
# Cell 5.4: Test Agentic Retrieval with Complex Question
#
# Now we can query using the Retrieve API. The knowledge base will:
# 1. Use the LLM to decompose the question into subqueries
# 2. Execute each subquery against the index (using the vectorizer!)
# 3. Semantic rerank and merge results
# 4. Return references with citations
#
# IMPORTANT: Agentic Retrieval requires Entra ID authentication (DefaultAzureCredential)
# API Keys do not work for the KnowledgeBaseRetrievalClient.

if AGENTIC_AVAILABLE and KNOWLEDGE_BASE_CREATED:
    try:
        # Force reload to get fresh imports
        import sys
        azure_mods = [k for k in sys.modules.keys() if k.startswith('azure.search')]
        for mod in azure_mods:
            del sys.modules[mod]
        
        # These imports are from azure.search.documents.knowledgebases module
        from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
        from azure.search.documents.knowledgebases.models import (
            KnowledgeBaseRetrievalRequest,
            KnowledgeBaseMessage,
            KnowledgeBaseMessageTextContent,
            SearchIndexKnowledgeSourceParams,
            KnowledgeRetrievalLowReasoningEffort,
        )
        from azure.identity import DefaultAzureCredential
        
        print("✅ Retrieval client classes imported successfully")
        print("🔍 Testing Agentic Retrieval with complex multi-part question...\n")
        
        # CRITICAL: Use DefaultAzureCredential (Entra ID), NOT API Key!
        # API Keys don't work for agentic retrieval
        entra_credential = DefaultAzureCredential()
        
        # Create retrieval client for the knowledge base
        retrieval_client = KnowledgeBaseRetrievalClient(
            endpoint=SEARCH_ENDPOINT,
            knowledge_base_name=KNOWLEDGE_BASE_NAME,
            credential=entra_credential  # Must use Entra ID!
        )
        
        # Complex multi-part question that benefits from query decomposition
        complex_question = """
        Tell me about Metro Station 36:
        1. Where exactly is the station located?
        2. What is the expected passenger capacity during peak hours?
        3. What are the land use types in the surrounding area?
        """
        
        print(f"❓ Question:\n{complex_question}")
        print("="*60)
        print("⏳ Processing... (LLM decomposes → parallel search → merge)")
        print("="*60 + "\n")
        
        # Create retrieval request following Microsoft's example
        retrieval_request = KnowledgeBaseRetrievalRequest(
            messages=[
                KnowledgeBaseMessage(
                    role="user",
                    content=[KnowledgeBaseMessageTextContent(text=complex_question)]
                )
            ],
            knowledge_source_params=[
                SearchIndexKnowledgeSourceParams(
                    knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                    include_references=True,
                    include_reference_source_data=True,
                    always_query_source=True
                )
            ],
            include_activity=True,  # Show subqueries generated
            retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
        )
        
        # Execute retrieval
        result = retrieval_client.retrieve(retrieval_request=retrieval_request)
        
        print("✅ Agentic Retrieval Complete!\n")
        
        # Show response (synthesized answer)
        if hasattr(result, 'response') and result.response:
            print("📝 Synthesized Answer:")
            for resp in result.response:
                for content in resp.content:
                    if hasattr(content, 'text'):
                        print(f"   {content.text[:500]}...")
            print()
        
        # Show activity (subqueries generated)
        if hasattr(result, 'activity') and result.activity:
            print("🔍 Activity (subqueries and operations):")
            for activity in result.activity:
                activity_dict = activity.as_dict() if hasattr(activity, 'as_dict') else activity
                if isinstance(activity_dict, dict):
                    act_type = activity_dict.get('type', 'unknown')
                    if act_type == 'searchIndex':
                        args = activity_dict.get('search_index_arguments', {})
                        query = args.get('search', 'N/A')
                        print(f"   • Search: {query[:60]}...")
                    elif act_type == 'modelQueryPlanning':
                        tokens = activity_dict.get('input_tokens', 0)
                        print(f"   • Query Planning: {tokens} input tokens")
                    elif act_type == 'modelAnswerSynthesis':
                        tokens = activity_dict.get('output_tokens', 0)
                        print(f"   • Answer Synthesis: {tokens} output tokens")
            print()
        
        # Show references found
        if hasattr(result, 'references') and result.references:
            print(f"📄 Retrieved {len(result.references)} references:")
            for i, ref in enumerate(result.references[:5], 1):
                ref_dict = ref.as_dict() if hasattr(ref, 'as_dict') else ref
                if isinstance(ref_dict, dict):
                    source = ref_dict.get('source_data', {})
                    ct = source.get('content_type', 'unknown')
                    section = source.get('section_header', '')
                    score = ref_dict.get('reranker_score', 0)
                    print(f"   [{i}] Type: {ct}, Section: {section}, Score: {score:.2f}")
        else:
            print("   No references returned")
            
        print("\n" + "="*60)
        print("💡 Agentic Retrieval automatically:")
        print("   1. Decomposed your complex question into focused subqueries")
        print("   2. Ran each subquery with vector search (using the vectorizer)")
        print("   3. Merged and reranked results semantically")
        print("   4. Synthesized a natural language answer with citations")
        print("="*60)
        
    except ImportError as e:
        print(f"⚠️  Import error - retrieval client classes not available.")
        print(f"   Error: {e}")
        print("\n   To fix, restart VS Code and re-run the notebook.")
    except Exception as e:
        error_str = str(e)
        print(f"❌ Retrieval error: {e}")
        
        if "Forbidden" in error_str or "403" in error_str:
            print("\n" + "="*60)
            print("🔐 PERMISSION ERROR - Additional RBAC roles needed")
            print("="*60)
            print("   Agentic Retrieval requires these roles on your Azure user:")
            print("   • Search Service Contributor")
            print("   • Search Index Data Contributor")
            print("   • Search Index Data Reader")
            print("\n   AND the Search service needs:")
            print("   • System-assigned managed identity enabled")
            print("   • Cognitive Services OpenAI User role on Azure OpenAI")
            print("\n   Steps to fix:")
            print("   1. Azure Portal → AI Search service → Settings → Identity")
            print("   2. Enable 'System assigned' managed identity")
            print("   3. Azure Portal → AI Search → Access control (IAM)")
            print("   4. Add the three Search roles to yourself")
            print("   5. Azure Portal → OpenAI resource → Access control (IAM)")
            print("   6. Add 'Cognitive Services OpenAI User' to the Search identity")
        else:
            print("\n   This may indicate:")
            print("   - Knowledge base not properly configured")
            print("   - Azure OpenAI connection issues")
            print("   - Region does not support agentic retrieval")
else:
    if not AGENTIC_AVAILABLE:
        print("⏭️  Skipping - Agentic Retrieval not available (Standard tier required).")
    elif not KNOWLEDGE_BASE_CREATED:
        print("⏭️  Skipping - Knowledge Base was not created (see cell 5.3).")
    
    print("\n" + "="*60)
    print("✅ MODULE 5 COMPLETE (Parts 1-4)")
    print("="*60)
    print("You've learned all the essential RAG retrieval concepts:")
    print("   • Embeddings and semantic similarity")
    print("   • Azure AI Search index design")
    print("   • Text, Vector, Hybrid, and Semantic search modes")
    print("   • Multi-retriever and filtered retrieval patterns")
    print("   • Complete RAG pipeline with GPT-4.1")
    print("\nAgentic Retrieval is an advanced preview feature that requires")
    print("additional RBAC configuration. Proceed to Module 6: GraphRAG!")

---

# 🎓 Summary

## What We Learned

1. **Embeddings** capture semantic meaning - English queries can find Hebrew content!
   - Cross-lingual similarity typically 0.5-0.7 (still very useful for retrieval)
   
2. **Azure AI Search** provides multiple search modes:
   - Text (BM25) - keyword matching, great for exact terms
   - Vector (kNN) - semantic similarity, great for meaning
   - Hybrid (RRF) - combines both, best for general RAG
   - Semantic (L2 reranker) - neural reranking for production quality
   
3. **Index design** matters: include `content_type` field for filtering

4. **Multi-retriever** patterns balance results across tables, figures, text

5. **Full RAG pipeline**: Embed → Index → Retrieve → Generate

6. **Two Types of Indexes** (Part 5):
   - **Normal Index**: Pre-computed embeddings, client-side query embedding
   - **Agentic Index**: Integrated vectorizer, auto-embeds at query time

## Index Comparison

| Feature | Normal Index (Parts 1-4) | Agentic Index (Part 5) |
|---------|--------------------------|------------------------|
| Index Name | `module5-metro-index` | `module5-metro-agentic` |
| Query Embedding | Client-side (manual) | Server-side (vectorizer) |
| Semantic Config | Named config | **Default** config |
| Description | Optional | Recommended (helps agent) |
| Use Case | Standard RAG | Complex multi-part questions |
| Tier Required | Any | Standard (S1+) |

## Key Takeaways

| Component | Recommendation |
|-----------|----------------|
| Embedding Model | `text-embedding-3-large` (3072d) |
| Search Mode | Hybrid + Semantic for production |
| Index Design | Include `content_type` field for filtering |
| Retrieval | Multi-retriever for mixed content types |
| Top-K | Start with 5, adjust based on results |
| Multilingual | Embeddings handle Hebrew ↔ English |
| Agentic | Use for complex compound questions |

## What If Part 5 Was Skipped?

If you saw "Skipping - Agentic Retrieval not available", that's **expected** for Basic tier Azure AI Search. You still learned the core concepts:
- Parts 1-4 cover everything needed for production RAG
- Agentic Retrieval is an advanced feature for complex multi-part questions
- Consider upgrading to Standard tier if you need Agentic Retrieval

---

## Next Steps

**Module 6: GraphRAG** - Cross-document reasoning with knowledge graphs

---

## Cleanup (Optional)

Run the cell below to delete the indexes and agentic resources if you want to start fresh:

In [ ]:
# Cell: Cleanup - Delete indexes and agentic resources (OPTIONAL)
# Uncomment and run if you want to delete the resources

cleanup_commands = """
# Delete normal index (Parts 1-4)
# index_client.delete_index("module5-metro-index")
# print("✅ Normal index deleted")

# Delete agentic resources (Part 5) - must delete in order!
# try:
#     index_client.delete_knowledge_base("module5-metro-kb")
#     print("✅ Knowledge Base deleted")
# except: pass

# try:
#     index_client.delete_knowledge_source("module5-metro-ks")
#     print("✅ Knowledge Source deleted")
# except: pass

# try:
#     index_client.delete_index("module5-metro-agentic")
#     print("✅ Agentic index deleted")
# except: pass
"""

print("💡 To cleanup, uncomment and run the commands above.")
print("\n📋 Resources created in this module:")
print(f"   - Normal Index: module5-metro-index")
print(f"   - Agentic Index: module5-metro-agentic (if Part 5 ran)")
print(f"   - Knowledge Source: module5-metro-ks (if Part 5 ran)")
print(f"   - Knowledge Base: module5-metro-kb (if Part 5 ran)")
print("\n⚠️  Delete in order: Knowledge Base → Knowledge Source → Index")